<a href="https://colab.research.google.com/github/Vadapao26/WiseWaste-Outward-Sales-Dispatch-Analytics/blob/main/Sales_Analytics_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WiseWaste-Outward-Sales-Dispatch-Analytics

In [ ]:
# ── STEP 1: Install (run once per session) ───────────────────────────────────
!pip install openpyxl --quiet
print("✅ openpyxl ready")


✅ openpyxl ready


In [ ]:
# ── STEP 2: Upload your Outward CSV ─────────────────────────────────────────
from google.colab import files
import io, pandas as pd

print("📂 Please upload your Outward CSV file...")
uploaded = files.upload()
_fname = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[_fname]))
print(f"✅ Loaded '{_fname}' — {len(df_raw)} rows × {len(df_raw.columns)} columns")


📂 Please upload your Outward CSV file...


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  — edit these values if needed
# ══════════════════════════════════════════════════════════════════════════════
import re, pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

OUTPUT_FILENAME         = "Sales_Analytics_Dashboard.xlsx"
REJECTION_THRESHOLD_PCT = 5   # rows above this % are highlighted red

# Vendor name normalisation  (lowercase regex → clean name)
# Add new entries here as new vendors appear in future exports
VENDOR_MAP = {
    r'gsj.*|gst transport.*|gjs.*':       'GSJ Logistics',
    r'gokul.*':                            'Gokul Transport',
    r'tiru.*|thiru.*':                     'Thiru / Tiru Transport',
    r'indian.*express.*':                  'Indian Express',
    r'jk.*cement.*|jkcement.*':            'JK Cement',
    r'mrf.*':                              'MRF Vehicle',
    r'mm.*trad.*|mmtrad.*':               'MM Traders',
    r'trash.*cash.*|trashtocash.*':        'Trash To Cash',
    r'kalam.*':                            'Kalam Enterprise',
    r'putul.*':                            'Putul Traders',
    r'lucky.*':                            'Lucky Enterprises',
    r'rudraya.*|rudrayya.*':              'Rudraya Wastech',
    r'thandav.*|tandav.*':                'Tandav G Traders',
    r'shafiya.*':                          'Shafiya Enterprises',
    r'landbell.*':                         'Landbell GFS India',
    r'ecohandlers.*|echohandlers.*':       'EcoHandlers',
    r'chandapura.*':                       'Chandapura TMC',
    r'madanayakanhalli.*':                 'Madanayakanhalli',
    r'doddatogur.*|doddathogur.*':         'Doddathogur GP',
    r'dalmia.*':                           'DalmiaCement',
    r'basavaraju.*':                       'Basavaraju',
    r'bharat.*':                           'Bharat Enterprises',
    r'swmpl.*':                            'SWMPL',
    r'sahana.*':                           'Sahana Enterprises',
    r'vishuddh.*':                         'Vishuddh Recycle',
    r'vinayak.*':                          'Vinayak Industries',
    r'hassan.*':                           'Hassan Traders',
    r'rerout.*':                           'Rerout Private Limited',
    r'regenplastics.*':                    'REGENPLASTICS',
    r'banyan.*':                           'Banyan Sustainable',
    r'srichakra.*':                        'Srichakra Polyplast',
    r'krishna.*':                          'Krishna Polymach',
    r'cipet.*':                            'CIPET (KCDC)',
    r'myse.*|mysechotch.*':               'MysEchotch Recycling',
    r'kgpcl.*':                            'KGPCL',
    r'e prakruti.*':                       'E Prakruti',
    r'hebbagodi.*':                        'Hebbagodi CMC',
    r'claritas.*':                         'Claritas Recyclers',
    r'lakshmi.*':                          'LAKSHMI ELECTRICALS',
    r'shaila.*':                           'SHAILA PLASTICS',
}

# Material → Broad Category (from PDF nomenclature)
BROAD_CAT = {
    # Paper
    'Paper_Carton Box':'Paper', 'Paper_Color Paper':'Paper',
    'Paper_White Paper':'Paper', 'Paper_Carton_Grade-1':'Paper',
    'Paper_Carton_Grade-2':'Paper', 'Paper_Carton Roll_Grade-1':'Paper',
    'Paper_Carton Roll_Grade-2':'Paper', 'Paper_Tissue_Grade-1':'Paper',
    'Paper_Tissue_Grade-2':'Paper', 'Paper_Paper Cup':'Paper',
    # Metal
    'Metal_Iron':'Metal', 'Metal_Aluminum_Can':'Metal',
    'Metal_Aluminum_Foil':'Metal', 'Metal_Aluminium_Others':'Metal',
    'Metal_Stainless steel':'Metal', 'Metal_Copper':'Metal',
    'Metal_Mild steel':'Metal', 'Metal_GI(Galvanized Iron)':'Metal',
    # Glass
    'Glass_Windshield':'Glass', 'Glass_Glass Bottle':'Glass',
    'Glass_Broken Glass':'Glass',
    # Rigid Plastics
    'Plastics_PETE_Bottle_Natural_Grade-1':'Rigid Plastics',
    'Plastics_PETE_Bottle_Natural_Grade-2':'Rigid Plastics',
    'Plastics_PETE_Bottle_Green':'Rigid Plastics',
    'Plastics_PETE_Bottle_Brown':'Rigid Plastics',
    'Plastics_PETE_Bottle_White':'Rigid Plastics',
    'Plastics_PETE_Others_Natural':'Rigid Plastics',
    'Plastics_PETE_Mixed color':'Rigid Plastics',
    'Plastics_HDPE_Red_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Green_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Blue_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Yellow_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_White_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Black_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Mixed color_Grade-1':'Rigid Plastics',
    'Plastics_HDPE_Grade 1_Blue':'Rigid Plastics',
    'Plastics_HDPE_Grade 1_White':'Rigid Plastics',
    'Plastics_HDPE_Grade 1_Mixed':'Rigid Plastics',
    'Plastics_HDPE_Red_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Green_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Blue_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Yellow_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_White_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Black_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Mixed color_Grade-2':'Rigid Plastics',
    'Plastics_HDPE_Cans':'Rigid Plastics', 'Plastic_HDPE_Drums':'Rigid Plastics',
    'Plastics_HDPE_FMCG':'Rigid Plastics', 'Plastics_HDPE_Non FMCG':'Rigid Plastics',
    'Plastics_HDPE_Brush':'Rigid Plastics',
    'Plastics_PVC_Pipe_Black':'Rigid Plastics', 'Plastics_PVC_Pipe_White':'Rigid Plastics',
    'Plastics_PVC_Pipe_Blue':'Rigid Plastics', 'Plastics_PVC_Pipe_Grey':'Rigid Plastics',
    'Plastics_PVC_Others_Natural':'Rigid Plastics', 'Plastics_PVC_Others':'Rigid Plastics',
    'Plastics_PP_Natural':'Rigid Plastics', 'Plastics_PP_White':'Rigid Plastics',
    'Plastics_PP_Black':'Rigid Plastics', 'Plastics_PP_Mixed':'Rigid Plastics',
    'Plastics_PP_Natural_Grade-1':'Rigid Plastics', 'Plastics_PP_White_Grade-1':'Rigid Plastics',
    'Plastics_PP_Black_Grade-1':'Rigid Plastics', 'Plastics_PP_Red_Grade-1':'Rigid Plastics',
    'Plastics_PP_Green_Grade-1':'Rigid Plastics', 'Plastics_PP_Blue_Grade-1':'Rigid Plastics',
    'Plastics_PP_Yellow_Grade-1':'Rigid Plastics', 'Plastics_PP_Mixed_Grade-1':'Rigid Plastics',
    'Plastics_PP_Ivory_Grade-1':'Rigid Plastics',
    'Plastics_PP_Natural_Grade-2':'Rigid Plastics', 'Plastics_PP_White_Grade-2':'Rigid Plastics',
    'Plastics_PP_Black_Grade-2':'Rigid Plastics', 'Plastics_PP_Red_Grade-2':'Rigid Plastics',
    'Plastics_PP_Green_Grade-2':'Rigid Plastics', 'Plastics_PP_Blue_Grade-2':'Rigid Plastics',
    'Plastics_PP_Yellow_Grade-2':'Rigid Plastics', 'Plastics_PP_Mixed_Grade-2':'Rigid Plastics',
    'Plastics_PP_Ivory_Grade-2':'Rigid Plastics',
    'Plastics_PS_Thermocol':'Rigid Plastics', 'Plastics_PS_Foam':'Rigid Plastics',
    'Plastics_PC':'Rigid Plastics', 'Plastics_ABS':'Rigid Plastics',
    'Plastics_PPCP Bucket':'Rigid Plastics', 'Plastics_PPCP Buckets':'Rigid Plastics',
    'Mixed Rigid Plastics':'Rigid Plastics',
    # Flexible Plastics
    'Plastics_LDPE_Grade-1':'Flexible Plastics', 'Plastics_LDPE_Grade-2':'Flexible Plastics',
    'Plastics_LDPE_Grade 1':'Flexible Plastics', 'Plastics_LDPE_Print/Colour':'Flexible Plastics',
    'Plastic_Rigid_LDPE':'Flexible Plastics', 'Plastics_LDPE_Foam':'Flexible Plastics',
    'Plastics_PE_Covers_Milk Pouches':'Flexible Plastics',
    'Plastics_PE_Covers_Oil Pouches':'Flexible Plastics',
    'Plastics_PE_HM':'Flexible Plastics', 'PP_Woven Bags':'Flexible Plastics',
    'Plastics_PP_Flexibles_Natural_Grade-1':'Flexible Plastics',
    'Plastic_PP_Flexibles_Mixed_Grade-1':'Flexible Plastics',
    'Plastics_PP_Flexibles_Natural_Grade-2':'Flexible Plastics',
    'Plastic_PP_Flexibles_Mixed_Grade-2':'Flexible Plastics',
    'Plastics_PP_Flexibles_Natural':'Flexible Plastics',
    'Plastics_MLP_Metalised':'Flexible Plastics', 'Plastics_MLP_Non Metalised':'Flexible Plastics',
    'Mixed Flexible Plastics':'Flexible Plastics', 'Plastics_Bale Straps':'Flexible Plastics',
    # LVP  (co-processing — zero sale price, revenue tracked at trip level)
    'Plastics_LVP_Rigid Plastic':'LVP', 'Plastics_LVP_Flexible plastics':'LVP',
    'LVP (Co-Processing)':'LVP',
    # Coconut Shell
    'Coconut Shell':'Coconut Shell', 'Tender coconut':'Coconut Shell',
    # Others
    'Plastics_TetraPak':'Others', 'Tetra Pak':'Others',
    'Wood':'Others', 'Footwear':'Others', 'Rubber':'Others',
    # SCF
    'SCF':'SCF', 'SCF_Textile':'SCF',
    # Rejects / zero-value outward
    'Rejects':'Rejects', 'Reject_Food contaminated waste':'Rejects',
    'Reject_Dry waste':'Rejects', 'Reject_Direct':'Rejects',
    'Drywaste_Unsorted':'Rejects', 'Unsorted Drywaste':'Rejects',
    # E-Waste
    'E-waste':'E-Waste',
    # Textile
    'Textile Waste':'Textile Waste',
    # Wet Waste
    'Garden waste':'Wet Waste', 'Compost':'Wet Waste', 'Food waste':'Wet Waste',
    # Sanitary
    'Sanitary waste':'Sanitary Waste',
    # C&D
    'C&D_POP':'C&D Waste', 'C&D_Gypsum Boards':'C&D Waste',
    'C&D_Ceramics':'C&D Waste', 'C&D_False Ceiling':'C&D Waste',
    'C&D_Tiles':'C&D Waste', 'C&D_Mixed debris':'C&D Waste',
    'C&D_Concrete':'C&D Waste', 'C&D_Others':'C&D Waste',
}

print("✅ Configuration loaded")


# ══════════════════════════════════════════════════════════════════════════════
# DATA CLEANING & ENRICHMENT
# ══════════════════════════════════════════════════════════════════════════════
df = df_raw.copy()

# Dates
df['Date'] = pd.to_datetime(df['Date'], format='%d %b %Y', errors='coerce')
df['MonthStr'] = df['Date'].dt.strftime('%b %Y')

# Numeric coercion
_num = ['Dispatched Qty (in KG)', 'Rate (in Rs)', 'Billed Amount (in Rs)',
        'Rejection Qty (in KG)', 'Value of Accepted Material (in Rs)', 'Rejection %',
        'Transport Cost (in Rs)', 'Loading Cost', 'Additional Transport Cost',
        'Incentive Cost/KG', 'Total Incentive Cost (in Rs)',
        'Total Value of accepted qty (in Rs)', 'Total Dispatched Qty (in KG)']
for c in _num:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# State name cleanup (Karnataka vs 'Karnataka ')
df['State'] = df['State'].str.strip().str.title()

# Derived material-level columns
df['AcceptedQty'] = df['Dispatched Qty (in KG)'] - df['Rejection Qty (in KG)']

# ── Revenue strategy ──────────────────────────────────────────────────────────
# Billed Amount (in Rs)               → what we billed per material (pre-rejection)
# Value of Accepted Material (in Rs)  → net per material after rejection (0 for LVP/Rejects/Textile)
# Total Value of accepted qty (in Rs) → TRIP-LEVEL total incl. LVP co-processing incentives
#
# For material-level revenue we use Billed Amount (most reliable per-material figure)
# For total trip revenue (incl. LVP incentive) we use Total Value of accepted qty deduped

# Vendor normalisation
def normalise_vendor(v):
    v_lower = str(v).strip().lower()
    for pattern, name in VENDOR_MAP.items():
        if re.fullmatch(pattern, v_lower):
            return name
    return str(v).strip().title()

df['VendorClean'] = df['Transport Vendor'].apply(normalise_vendor)

# Broad category
def get_broad(m):
    if m in BROAD_CAT:
        return BROAD_CAT[m]
    ml = m.lower()
    if 'paper' in ml:                                                       return 'Paper'
    if any(x in ml for x in ['metal','iron','steel','copper','alumin']):   return 'Metal'
    if 'glass' in ml:                                                       return 'Glass'
    if any(x in ml for x in ['hdpe','pete','ppcp','ps_','pvc','_pc','_abs']): return 'Rigid Plastics'
    if any(x in ml for x in ['ldpe','pe_','flexible','woven bag','mlp']): return 'Flexible Plastics'
    if 'lvp' in ml:                                                         return 'LVP'
    if 'reject' in ml or 'unsorted' in ml:                                 return 'Rejects'
    if 'e-waste' in ml or 'ewaste' in ml:                                  return 'E-Waste'
    if 'textile' in ml:                                                     return 'Textile Waste'
    if any(x in ml for x in ['coconut','wet waste','garden','compost']):   return 'Wet Waste'
    if 'scf' in ml:                                                         return 'SCF'
    if 'c&d' in ml or 'debris' in ml:                                      return 'C&D Waste'
    return 'Others'

df['BroadCategory'] = df['Material'].apply(get_broad)

# ── Trip-level deduplication ──────────────────────────────────────────────────
# ONLY used for columns that repeat the same value across all rows of a trip:
# Transport Cost, Loading Cost, Additional Transport Cost,
# Total Incentive Cost, Total Value of accepted qty, Payment Status
df_trips = df.drop_duplicates(subset='Outward Code').copy()
df_trips['TotalTransportCost'] = (
    df_trips['Transport Cost (in Rs)'] +
    df_trips['Loading Cost'] +
    df_trips['Additional Transport Cost']
)
# LVP incentive = trip total revenue - sum of per-material billed amounts
_trip_billed = df.groupby('Outward Code')['Billed Amount (in Rs)'].sum().reset_index(name='SumBilledPerTrip')
df_trips = df_trips.merge(_trip_billed, on='Outward Code', how='left')
df_trips['LVP_Incentive'] = (
    df_trips['Total Value of accepted qty (in Rs)'] - df_trips['SumBilledPerTrip']
).clip(lower=0)

# Merge transport cost back to row-level for per-row lookups
df = df.merge(df_trips[['Outward Code','TotalTransportCost','LVP_Incentive']], on='Outward Code', how='left')

# ── Sanity check printout ─────────────────────────────────────────────────────
paid_set   = set(df_trips[df_trips['Payment Status']==True]['Outward Code'])
unpaid_set = set(df_trips[df_trips['Payment Status']==False]['Outward Code'])

print(f"\n✅ Data cleaned — {len(df)} material rows | {df['Outward Code'].nunique()} unique trips")
print(f"   Date range  : {df['Date'].min().strftime('%d %b %Y')} → {df['Date'].max().strftime('%d %b %Y')}")
print(f"   Materials   : {df['Material'].nunique()} unique")
print(f"   Customers   : {df['Customer'].nunique()} unique")
print(f"   Vendors     : {df['VendorClean'].nunique()} (after normalisation)")
print(f"   States      : {df['State'].nunique()} ({', '.join(sorted(df['State'].unique()))})")
print()
print("── Revenue Summary ──────────────────────────────────────────────────────")
print(f"   Total Billed (material sales)   : ₹{df['Billed Amount (in Rs)'].sum():>14,.2f}")
print(f"   Net (after rejection deduction) : ₹{df['Value of Accepted Material (in Rs)'].sum():>14,.2f}")
print(f"   LVP/Co-processing incentive     : ₹{df_trips['LVP_Incentive'].sum():>14,.2f}")
print(f"   Grand Total (incl. LVP)         : ₹{df_trips['Total Value of accepted qty (in Rs)'].sum():>14,.2f}")
print()
print("── Operations ───────────────────────────────────────────────────────────")
print(f"   Total Dispatched KG  : {df['Dispatched Qty (in KG)'].sum():>14,.2f}")
print(f"   Total Accepted KG    : {df['AcceptedQty'].sum():>14,.2f}")
print(f"   Total Rejected KG    : {df['Rejection Qty (in KG)'].sum():>14,.2f}")
print(f"   Overall Rejection %  : {df['Rejection Qty (in KG)'].sum()/max(df['Dispatched Qty (in KG)'].sum(),1)*100:>13.2f}%")
print(f"   Total Transport Cost : ₹{df_trips['TotalTransportCost'].sum():>14,.2f}")
print(f"   Total Incentive Cost : ₹{df_trips['Total Incentive Cost (in Rs)'].sum():>14,.2f}")
print(f"   Paid trips           : {len(paid_set):>14} / {df['Outward Code'].nunique()}")
print(f"   Unpaid trips         : {len(unpaid_set):>14}")
unpaid_rev = df[df['Outward Code'].isin(unpaid_set)]['Billed Amount (in Rs)'].sum()
print(f"   Unpaid Revenue       : ₹{unpaid_rev:>14,.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# EXCEL STYLING HELPERS
# ══════════════════════════════════════════════════════════════════════════════
C_NAVY   = "1F3864"; C_BLUE   = "2E75B6"; C_LBLUE  = "D6E4F0"
C_GREEN  = "E8F5E9"; C_AMBER  = "FFF3CD"; C_ROSE   = "FCE4EC"
C_WHITE  = "FFFFFF"; C_TOTAL  = "FFF2CC"; C_SUB    = "BDD7EE"
C_RED_H  = "C00000"; C_RED_R  = "FFE0E0"; C_UNPAID = "FFCCCC"
C_GREEN_H= "375623"; C_BROWN  = "7B3F00"; C_PURPLE = "4A0072"

_ts = Side(style="thin",   color="AAAAAA")
_tk = Side(style="medium", color="1F3864")
def _tb(): return Border(left=_ts, right=_ts, top=_ts, bottom=_ts)
def _bk(): return Border(left=_tk, right=_tk, top=_tk, bottom=_tk)
def _hf(sz=11, col=C_WHITE): return Font(name="Arial", bold=True, size=sz, color=col)
def _df(sz=10):               return Font(name="Arial", size=sz)
def _bf(sz=10, col="000000"):return Font(name="Arial", bold=True, size=sz, color=col)
def _fl(col): return PatternFill("solid", fgColor=col)
def _ctr(): return Alignment(horizontal="center", vertical="center", wrap_text=True)
def _lft(): return Alignment(horizontal="left",   vertical="center", wrap_text=True)
def _rgt(): return Alignment(horizontal="right",  vertical="center")

def hdr_row(ws, row, ncols, bg=C_NAVY, fg=C_WHITE, h=28):
    ws.row_dimensions[row].height = h
    for c in range(1, ncols+1):
        cell = ws.cell(row, c)
        cell.font=_hf(col=fg); cell.fill=_fl(bg); cell.alignment=_ctr(); cell.border=_tb()

def total_row(ws, row, ncols):
    for c in range(1, ncols+1):
        cell = ws.cell(row, c)
        cell.font=_bf(col=C_NAVY); cell.fill=_fl(C_TOTAL)
        cell.border=_bk(); cell.alignment=_rgt() if c>1 else _lft()

def set_w(ws, widths):
    for i,w in enumerate(widths,1): ws.column_dimensions[get_column_letter(i)].width=w

def title_block(ws, title, sub, ncols):
    ws.row_dimensions[1].height=36; ws.row_dimensions[2].height=20
    ws.merge_cells(start_row=1,start_column=1,end_row=1,end_column=ncols)
    t=ws.cell(1,1,title)
    t.font=Font(name="Arial",bold=True,size=16,color=C_WHITE)
    t.fill=_fl(C_NAVY); t.alignment=_ctr()
    ws.merge_cells(start_row=2,start_column=1,end_row=2,end_column=ncols)
    s=ws.cell(2,1,sub)
    s.font=Font(name="Arial",italic=True,size=10,color="444444")
    s.fill=_fl("D9E1F2"); s.alignment=_ctr()

def sec(ws, row, text, ncols, bg=C_SUB, fg=C_NAVY):
    c=ws.cell(row,1,text); c.font=_bf(11,fg); c.fill=_fl(bg)
    if ncols>1: ws.merge_cells(start_row=row,start_column=1,end_row=row,end_column=ncols)

def wr(ws, row, vals, fill, fmts=None):
    for ci,v in enumerate(vals,1):
        c=ws.cell(row,ci, round(v,4) if isinstance(v,float) else v)
        c.fill=_fl(fill); c.border=_tb(); c.font=_df()
        c.alignment=_rgt() if ci>1 else _lft()
        if fmts and ci in fmts: c.number_format=fmts[ci]

def totrow_formula(ws, tr, ncols, col_fmts):
    total_row(ws, tr, ncols)
    ws.cell(tr,1,"TOTAL")
    for cl,fmt in col_fmts:
        ci=ord(cl)-64
        ws.cell(tr,ci,f'=SUM({cl}{tr-len(list(ws.iter_rows(min_row=4,max_row=tr-1)))+1}:{cl}{tr-1})').number_format=fmt

# Number format constants
INR='₹#,##0.00'; NUM='#,##0.00'; PCT='0.00%'; INT='#,##0'

# Alternating row fill
def rfill(r, alt1=C_GREEN, alt2=C_WHITE): return alt1 if r%2==1 else alt2

print("✅ Styling helpers ready")


# ══════════════════════════════════════════════════════════════════════════════
# PRE-COMPUTE SHARED AGGREGATES
# ══════════════════════════════════════════════════════════════════════════════
# Trip-level totals (all from df_trips — deduped)
_tt  = df_trips['TotalTransportCost'].sum()
_ti  = df_trips['Total Incentive Cost (in Rs)'].sum()
_lvp = df_trips['LVP_Incentive'].sum()
_paid_n   = len(paid_set)
_unpaid_n = len(unpaid_set)
_total_trip_rev = df_trips['Total Value of accepted qty (in Rs)'].sum()

# Material-level totals (from df — full rows)
_disp  = df['Dispatched Qty (in KG)'].sum()
_acc   = df['AcceptedQty'].sum()
_rej   = df['Rejection Qty (in KG)'].sum()
_billed= df['Billed Amount (in Rs)'].sum()
_netmat= df['Value of Accepted Material (in Rs)'].sum()
_unpaid_rev = df[df['Outward Code'].isin(unpaid_set)]['Billed Amount (in Rs)'].sum()
_n_trips = df['Outward Code'].nunique()

wb = Workbook()
wb.remove(wb.active)
print("✅ Workbook created — building sheets...")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 1: SUMMARY DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("📊 Summary Dashboard")
title_block(ws, "Sales Analytics — Summary Dashboard",
    f"Data: {df['Date'].min().strftime('%d %b %Y')} → {df['Date'].max().strftime('%d %b %Y')}"
    f"   |   {_n_trips} trips   |   {len(df)} material rows", 8)

# KPI table
kpis = [
    ("── VOLUME ──────────────────", "",  "",  ""),
    ("Total Dispatched (KG)",  _disp,                       NUM,  ""),
    ("Total Accepted (KG)",    _acc,                         NUM,  ""),
    ("Total Rejected (KG)",    _rej,                         NUM,  ""),
    ("Overall Rejection %",    _rej/max(_disp,1)*100,       '0.00"%"', f"Target < {REJECTION_THRESHOLD_PCT}%"),
    ("── REVENUE ─────────────────", "",  "",  ""),
    ("Total Billed (₹)",       _billed,                      INR,  "Material sales"),
    ("Net Revenue (₹)",        _netmat,                      INR,  "After rejection deduction"),
    ("LVP Co-processing (₹)",  _lvp,                         INR,  "Incentive from recyclers"),
    ("Grand Total Revenue (₹)",_total_trip_rev,              INR,  "Billed + LVP incentive"),
    ("── COSTS ───────────────────", "",  "",  ""),
    ("Total Transport Cost (₹)",_tt,                         INR,  "Loading + freight + addl"),
    ("Total Incentive Cost (₹)",_ti,                         INR,  ""),
    ("── PAYMENTS ────────────────", "",  "",  ""),
    ("Total Trips",            _n_trips,                     INT,  ""),
    ("Paid Trips",             _paid_n,                      INT,  f"of {_n_trips} trips"),
    ("Unpaid Trips",           _unpaid_n,                    INT,  "⚠️ Follow up"),
    ("Unpaid Billed Revenue (₹)",_unpaid_rev,                INR,  "Outstanding"),
    ("── MASTER DATA ─────────────", "",  "",  ""),
    ("Unique Customers",       df['Customer'].nunique(),      INT,  ""),
    ("Unique Materials",       df['Material'].nunique(),      INT,  ""),
    ("Unique Transport Vendors",df['VendorClean'].nunique(),  INT,  "After name normalisation"),
]

sec(ws, 4, "KEY PERFORMANCE INDICATORS", 3)
for ci,h in enumerate(["Metric","Value","Notes"],1): ws.cell(5,ci,h)
hdr_row(ws, 5, 3, bg=C_BLUE, h=22)

r=6
for label,val,fmt,note in kpis:
    if label.startswith("──"):
        ws.merge_cells(start_row=r,start_column=1,end_row=r,end_column=3)
        c=ws.cell(r,1,label)
        c.font=_bf(9,C_NAVY); c.fill=_fl(C_SUB); c.alignment=_lft()
        for ci in range(1,4):
            ws.cell(r,ci).border=_tb()
    else:
        fill = C_GREEN if r%2==1 else C_WHITE
        v = round(val,2) if isinstance(val,float) else val
        for ci,cv in enumerate([label,v,note],1):
            c=ws.cell(r,ci,cv); c.fill=_fl(fill); c.font=_df()
            c.border=_tb(); c.alignment=_rgt() if ci==2 else _lft()
        ws.cell(r,2).number_format=fmt
    r+=1

# Monthly trend (right side, cols 5-8)
mg = (df.groupby('MonthStr')
        .agg(Trips=('Outward Code','nunique'),
             DispKG=('Dispatched Qty (in KG)','sum'),
             Billed=('Billed Amount (in Rs)','sum'),
             NetRev=('Value of Accepted Material (in Rs)','sum'))
        .reset_index())
mg['_d']=pd.to_datetime(mg['MonthStr'],format='%b %Y')
mg=mg.sort_values('_d').drop('_d',axis=1)
# add LVP per month
mg_lvp=(df_trips.groupby(df_trips['Date'].dt.strftime('%b %Y'))['LVP_Incentive']
          .sum().reset_index().rename(columns={'Date':'MonthStr'}))
mg=mg.merge(mg_lvp,on='MonthStr',how='left').fillna(0)
mg['GrandTotal']=mg['Billed']+mg['LVP_Incentive']

ws.cell(4,5,"MONTHLY PERFORMANCE").font=_bf(11,C_NAVY)
ws.cell(4,5).fill=_fl(C_SUB); ws.merge_cells(start_row=4,start_column=5,end_row=4,end_column=9)
for ci,h in enumerate(["Month","# Trips","Dispatched (KG)","Billed Rev (₹)","Grand Total (₹)"],5):
    c=ws.cell(5,ci,h); c.font=_hf(); c.fill=_fl(C_BLUE); c.alignment=_ctr(); c.border=_tb()

for i,(_, rd) in enumerate(mg.iterrows()):
    rr=6+i; fill=C_AMBER if rr%2==0 else C_WHITE
    for ci,v in enumerate([rd['MonthStr'],int(rd['Trips']),round(rd['DispKG'],2),
                            round(rd['Billed'],2),round(rd['GrandTotal'],2)],5):
        c=ws.cell(rr,ci,v); c.fill=_fl(fill); c.border=_tb(); c.font=_df()
        c.alignment=_rgt() if ci>5 else _lft()
    ws.cell(rr,7).number_format=NUM
    ws.cell(rr,8).number_format=INR
    ws.cell(rr,9).number_format=INR

set_w(ws,[28,18,22,6, 14,10,18,18,18])
print("   ✅ Sheet 1: Summary Dashboard")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 2: MATERIAL SALES
# ══════════════════════════════════════════════════════════════════════════════
ws2=wb.create_sheet("📦 Material Sales")
title_block(ws2,"Material Sales Analysis",
    "Ranked by Billed Revenue. Zero-price materials (LVP, Rejects, Textile) shown separately.",9)

mat=(df.groupby(['BroadCategory','Material','Type'])
       .agg(Trips=('Outward Code','nunique'),
            DispKG=('Dispatched Qty (in KG)','sum'),
            AccKG=('AcceptedQty','sum'),
            RejKG=('Rejection Qty (in KG)','sum'),
            RejPct=('Rejection %','mean'),
            AvgRate=('Rate (in Rs)','mean'),
            Billed=('Billed Amount (in Rs)','sum'),
            NetRev=('Value of Accepted Material (in Rs)','sum'))
       .reset_index().sort_values('Billed',ascending=False))

hdrs=["Broad Category","Material","Type","# Trips","Dispatched (KG)",
      "Accepted (KG)","Rejected (KG)","Avg Rej %","Avg Rate (₹/KG)","Billed Rev (₹)"]
for ci,h in enumerate(hdrs,1): ws2.cell(4,ci,h)
hdr_row(ws2,4,len(hdrs))

for i,(_, rd) in enumerate(mat.iterrows()):
    rr=5+i; fill=rfill(rr)
    wr(ws2,rr,[rd['BroadCategory'],rd['Material'],rd['Type'],rd['Trips'],
               rd['DispKG'],rd['AccKG'],rd['RejKG'],rd['RejPct']/100,
               rd['AvgRate'],rd['Billed']],
       fill,{5:NUM,6:NUM,7:NUM,8:PCT,9:INR,10:INR})

tr2=5+len(mat); total_row(ws2,tr2,len(hdrs)); ws2.cell(tr2,1,"TOTAL")
for cl,fmt in [('D',INT),('E',NUM),('F',NUM),('G',NUM),('J',INR)]:
    ci=ord(cl)-64
    ws2.cell(tr2,ci,f'=SUM({cl}5:{cl}{tr2-1})').number_format=fmt

# Category rollup section
sr=tr2+2; sec(ws2,sr,"REVENUE BY BROAD CATEGORY",9); sr+=1
cat_h=["Broad Category","# Trips","Dispatched (KG)","Accepted (KG)",
       "Rejected (KG)","Avg Rej %","Billed Rev (₹)","Rev Share %"]
for ci,h in enumerate(cat_h,1): ws2.cell(sr,ci,h)
hdr_row(ws2,sr,len(cat_h),bg=C_BLUE)

cat_g=(df.groupby('BroadCategory')
         .agg(T=('Outward Code','nunique'),
              D=('Dispatched Qty (in KG)','sum'),
              A=('AcceptedQty','sum'),
              R=('Rejection Qty (in KG)','sum'),
              RP=('Rejection %','mean'),
              B=('Billed Amount (in Rs)','sum'))
         .reset_index().sort_values('B',ascending=False))
tb=cat_g['B'].sum()
for i,(_, rd) in enumerate(cat_g.iterrows()):
    rr=sr+1+i; fill=C_ROSE if rr%2==0 else C_WHITE
    wr(ws2,rr,[rd['BroadCategory'],rd['T'],rd['D'],rd['A'],
               rd['R'],rd['RP']/100,rd['B'],rd['B']/tb if tb else 0],
       fill,{3:NUM,4:NUM,5:NUM,6:PCT,7:INR,8:PCT})

set_w(ws2,[22,30,12,10,16,15,14,12,16,18])
ws2.freeze_panes='A5'
print("   ✅ Sheet 2: Material Sales")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 3: CUSTOMER ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
ws3=wb.create_sheet("🏢 Customer Analysis")
title_block(ws3,"Customer Business Analysis",
    "Revenue = Billed Amount. Transport & Incentive are trip-deduped per customer.",14)

c_mat=(df.groupby('Customer')
         .agg(Trips=('Outward Code','nunique'),
              DispKG=('Dispatched Qty (in KG)','sum'),
              AccKG=('AcceptedQty','sum'),
              RejKG=('Rejection Qty (in KG)','sum'),
              RejPct=('Rejection %','mean'),
              Billed=('Billed Amount (in Rs)','sum'),
              NetRev=('Value of Accepted Material (in Rs)','sum'))
         .reset_index())
c_trip=(df_trips.groupby('Customer')
                .agg(PaidT=('Payment Status','sum'),
                     Incentive=('Total Incentive Cost (in Rs)','sum'),
                     Transport=('TotalTransportCost','sum'),
                     LVPInc=('LVP_Incentive','sum'))
                .reset_index())
c_trip['PaidT']=c_trip['PaidT'].astype(int)
cust=c_mat.merge(c_trip,on='Customer',how='left').fillna(0)
cust['UnpaidT']=cust['Trips']-cust['PaidT']
cust['RevShare']=cust['Billed']/cust['Billed'].sum()
cust['GrandRev']=cust['Billed']+cust['LVPInc']
cust=cust.sort_values('GrandRev',ascending=False)

hdrs3=["Customer","# Trips","Dispatched (KG)","Accepted (KG)","Rejected (KG)",
       "Avg Rej %","Billed Rev (₹)","LVP Incentive (₹)","Grand Rev (₹)","Rev Share %",
       "Paid","Unpaid","Incentive Cost (₹)","Transport Cost (₹)"]
for ci,h in enumerate(hdrs3,1): ws3.cell(4,ci,h)
hdr_row(ws3,4,len(hdrs3))

for i,(_, rd) in enumerate(cust.iterrows()):
    rr=5+i; fill=rfill(rr)
    wr(ws3,rr,[rd['Customer'],rd['Trips'],rd['DispKG'],rd['AccKG'],rd['RejKG'],
               rd['RejPct']/100,rd['Billed'],rd['LVPInc'],rd['GrandRev'],
               rd['RevShare'],int(rd['PaidT']),int(rd['UnpaidT']),
               rd['Incentive'],rd['Transport']],
       fill,{3:NUM,4:NUM,5:NUM,6:PCT,7:INR,8:INR,9:INR,10:PCT,13:INR,14:INR})
    if rd['UnpaidT']>0: ws3.cell(rr,12).fill=_fl(C_UNPAID)

tr3=5+len(cust); total_row(ws3,tr3,len(hdrs3)); ws3.cell(tr3,1,"TOTAL")
for cl,fmt in [('B',INT),('C',NUM),('D',NUM),('E',NUM),
               ('G',INR),('H',INR),('I',INR),('M',INR),('N',INR)]:
    ci=ord(cl)-64
    ws3.cell(tr3,ci,f'=SUM({cl}5:{cl}{tr3-1})').number_format=fmt

set_w(ws3,[36,8,16,15,14,10,16,16,16,11,7,8,16,16])
ws3.freeze_panes='A5'
print("   ✅ Sheet 3: Customer Analysis")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 4: REJECTION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
ws4=wb.create_sheet("⚠️ Rejection Analysis")
title_block(ws4,"Rejection Analysis",
    f"Red rows = actual rejection > {REJECTION_THRESHOLD_PCT}%. Value Lost = Billed − Net Accepted.",8)

# By Customer
sec(ws4,4,"REJECTION BY CUSTOMER  (sorted by Actual Rejection %)",7);
hrj=["Customer","# Trips","Dispatched (KG)","Rejected (KG)","Avg Reported Rej %","Actual Rej %","Value Lost (₹)"]
for ci,h in enumerate(hrj,1): ws4.cell(5,ci,h)
hdr_row(ws4,5,len(hrj),bg=C_RED_H)

rc=(df.groupby('Customer')
      .agg(Trips=('Outward Code','nunique'),
           D=('Dispatched Qty (in KG)','sum'),
           R=('Rejection Qty (in KG)','sum'),
           RP=('Rejection %','mean'),
           B=('Billed Amount (in Rs)','sum'),
           N=('Value of Accepted Material (in Rs)','sum'))
      .reset_index())
rc['ActRej']=rc['R']/rc['D'].replace(0,1)
rc['VLost']=rc['B']-rc['N']
rc=rc.sort_values('ActRej',ascending=False)

for i,(_, rd) in enumerate(rc.iterrows()):
    rr=6+i
    fill=C_RED_R if rd['ActRej']*100>REJECTION_THRESHOLD_PCT else rfill(rr)
    wr(ws4,rr,[rd['Customer'],rd['Trips'],rd['D'],rd['R'],rd['RP']/100,rd['ActRej'],rd['VLost']],
       fill,{3:NUM,4:NUM,5:PCT,6:PCT,7:INR})

# By Material
mr=6+len(rc)+2; sec(ws4,mr,"REJECTION BY MATERIAL  (sorted by Rejected KG)",7); mr+=1
for ci,h in enumerate(hrj[:6]+[""],1): ws4.cell(mr,ci,h)
hdr_row(ws4,mr,len(hrj),bg=C_RED_H)

rm=(df[df['Rejection Qty (in KG)']>0]
      .groupby('Material')
      .agg(Trips=('Outward Code','nunique'),
           D=('Dispatched Qty (in KG)','sum'),
           R=('Rejection Qty (in KG)','sum'),
           RP=('Rejection %','mean'))
      .reset_index())
rm['ActRej']=rm['R']/rm['D'].replace(0,1)
rm=rm.sort_values('R',ascending=False)

for i,(_, rd) in enumerate(rm.iterrows()):
    rr=mr+1+i
    fill=C_RED_R if rd['ActRej']*100>REJECTION_THRESHOLD_PCT else rfill(rr)
    wr(ws4,rr,[rd['Material'],rd['Trips'],rd['D'],rd['R'],rd['RP']/100,rd['ActRej'],""],
       fill,{3:NUM,4:NUM,5:PCT,6:PCT})

set_w(ws4,[36,8,16,16,18,14,18])
print("   ✅ Sheet 4: Rejection Analysis")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 5: INCENTIVE COST
# ══════════════════════════════════════════════════════════════════════════════
ws5=wb.create_sheet("💰 Incentive Cost")
title_block(ws5,"Incentive Cost Analysis",
    "Incentive Cost/KG is material-level. Total Incentive Cost is trip-level (deduped).",7)

df['MatIncentive']=df['Incentive Cost/KG']*df['AcceptedQty']

# By material
sec(ws5,4,"INCENTIVE EARNED BY MATERIAL  (Incentive Rate × Accepted KG)",7)
h5=["Material","# Trips","Accepted (KG)","Rate (₹/KG)","Calculated Incentive (₹)","Billed Rev (₹)","Incentive as % Rev"]
for ci,h in enumerate(h5,1): ws5.cell(5,ci,h)
hdr_row(ws5,5,len(h5),bg=C_GREEN_H)

im=(df[df['Incentive Cost/KG']>0]
      .groupby('Material')
      .agg(Trips=('Outward Code','nunique'),
           AccKG=('AcceptedQty','sum'),
           Rate=('Incentive Cost/KG','mean'),
           Inc=('MatIncentive','sum'),
           Rev=('Billed Amount (in Rs)','sum'))
      .reset_index().sort_values('Inc',ascending=False))
im['Pct']=im['Inc']/im['Rev'].replace(0,1)

for i,(_, rd) in enumerate(im.iterrows()):
    rr=6+i; fill=rfill(rr,C_GREEN,C_WHITE)
    wr(ws5,rr,[rd['Material'],rd['Trips'],rd['AccKG'],rd['Rate'],rd['Inc'],rd['Rev'],rd['Pct']],
       fill,{3:NUM,4:INR,5:INR,6:INR,7:PCT})

tr5=6+len(im); total_row(ws5,tr5,len(h5)); ws5.cell(tr5,1,"TOTAL")
for cl,fmt in [('E',INR),('F',INR)]:
    ci=ord(cl)-64
    ws5.cell(tr5,ci,f'=SUM({cl}6:{cl}{tr5-1})').number_format=fmt

# By customer (trip-deduped)
cr=tr5+2; sec(ws5,cr,"INCENTIVE COST BY CUSTOMER  (Trip-deduped)",6); cr+=1
h5c=["Customer","# Trips","Total Incentive Cost (₹)","Billed Rev (₹)","Incentive as % Rev",""]
for ci,h in enumerate(h5c,1): ws5.cell(cr,ci,h)
hdr_row(ws5,cr,len(h5c),bg=C_GREEN_H)

ic=(df_trips[df_trips['Total Incentive Cost (in Rs)']>0]
      .groupby('Customer')
      .agg(T=('Outward Code','count'),
           Inc=('Total Incentive Cost (in Rs)','sum'))
      .reset_index())
ic_rev=df.groupby('Customer')['Billed Amount (in Rs)'].sum().reset_index(name='Rev')
ic=ic.merge(ic_rev,on='Customer',how='left')
ic['Pct']=ic['Inc']/ic['Rev'].replace(0,1)
ic=ic.sort_values('Inc',ascending=False)

for i,(_, rd) in enumerate(ic.iterrows()):
    rr=cr+1+i; fill=rfill(rr,C_GREEN,C_WHITE)
    wr(ws5,rr,[rd['Customer'],rd['T'],rd['Inc'],rd['Rev'],rd['Pct'],""],
       fill,{3:INR,4:INR,5:PCT})

set_w(ws5,[32,8,22,20,18,10])
print("   ✅ Sheet 5: Incentive Cost")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 6: TRANSPORT ANALYTICS
# ══════════════════════════════════════════════════════════════════════════════
ws6=wb.create_sheet("🚛 Transport Analytics")
title_block(ws6,"Transport & Logistics Analytics",
    "All cost figures are trip-deduped — no inflation from multi-material rows.",9)

# Vendor analysis (from df_trips)
sec(ws6,4,"VENDOR ANALYSIS  (trip-deduped costs)",9)
h6=["Vendor","# Trips","Dispatched (KG)","Loading (₹)","Additional (₹)","Freight (₹)","Total Cost (₹)","Cost/KG (₹)","Vehicles Used"]
for ci,h in enumerate(h6,1): ws6.cell(5,ci,h)
hdr_row(ws6,5,len(h6),bg=C_BROWN)

tv=(df_trips.groupby('VendorClean')
            .agg(Trips=('Outward Code','count'),
                 DispKG=('Total Dispatched Qty (in KG)','sum'),
                 Load=('Loading Cost','sum'),
                 Addl=('Additional Transport Cost','sum'),
                 Freight=('Transport Cost (in Rs)','sum'),
                 Total=('TotalTransportCost','sum'),
                 Veh=('Vehicle number','nunique'))
            .reset_index().sort_values('Total',ascending=False))
tv['CpKG']=tv['Total']/tv['DispKG'].replace(0,1)

for i,(_, rd) in enumerate(tv.iterrows()):
    rr=6+i; fill=rfill(rr,C_AMBER,C_WHITE)
    wr(ws6,rr,[rd['VendorClean'],rd['Trips'],rd['DispKG'],rd['Load'],rd['Addl'],
               rd['Freight'],rd['Total'],rd['CpKG'],rd['Veh']],
       fill,{3:NUM,4:INR,5:INR,6:INR,7:INR,8:INR})

tr6=6+len(tv); total_row(ws6,tr6,len(h6)); ws6.cell(tr6,1,"TOTAL")
for cl,fmt in [('B',INT),('C',NUM),('D',INR),('E',INR),('F',INR),('G',INR)]:
    ci=ord(cl)-64
    ws6.cell(tr6,ci,f'=SUM({cl}6:{cl}{tr6-1})').number_format=fmt

# Vehicle analysis
vr=tr6+2; sec(ws6,vr,"VEHICLE USAGE  (trip-deduped)",6); vr+=1
h6v=["Vehicle No.","State","# Trips","Primary Vendor","Dispatched (KG)","Transport Cost (₹)"]
for ci,h in enumerate(h6v,1): ws6.cell(vr,ci,h)
hdr_row(ws6,vr,len(h6v),bg=C_BROWN)

# Clean vehicle state
df_trips['VehState']=df_trips['Vehicle number'].str[:2].str.upper()
veh=(df_trips.groupby('Vehicle number')
             .agg(VehSt=('VehState','first'),
                  Trips=('Outward Code','count'),
                  Vendor=('VendorClean',lambda x: x.mode()[0] if not x.mode().empty else ''),
                  DispKG=('Total Dispatched Qty (in KG)','sum'),
                  TC=('TotalTransportCost','sum'))
             .reset_index().sort_values('Trips',ascending=False))

for i,(_, rd) in enumerate(veh.iterrows()):
    rr=vr+1+i; fill=rfill(rr,C_AMBER,C_WHITE)
    wr(ws6,rr,[rd['Vehicle number'],rd['VehSt'],rd['Trips'],rd['Vendor'],rd['DispKG'],rd['TC']],
       fill,{5:NUM,6:INR})
    ws6.cell(rr,1).alignment=_lft()
    ws6.cell(rr,3).alignment=_lft()

# Cost split summary
cr6=vr+len(veh)+2; sec(ws6,cr6,"TRANSPORT COST SPLIT",4); cr6+=1
for ci,h in enumerate(["Cost Component","Total (₹)","% of Total",""],1): ws6.cell(cr6,ci,h)
hdr_row(ws6,cr6,4,bg=C_BROWN)
ttc=df_trips['TotalTransportCost'].sum()
for i,(label,val) in enumerate([
    ("Loading Cost",       df_trips['Loading Cost'].sum()),
    ("Additional Cost",    df_trips['Additional Transport Cost'].sum()),
    ("Freight/Transport",  df_trips['Transport Cost (in Rs)'].sum()),
]):
    rr=cr6+1+i; fill=rfill(rr,C_AMBER,C_WHITE)
    wr(ws6,rr,[label,val,val/ttc if ttc else 0,""],fill,{2:INR,3:PCT})
total_row(ws6,cr6+4,4); ws6.cell(cr6+4,1,"TOTAL TRANSPORT")
ws6.cell(cr6+4,2,ttc).number_format=INR

set_w(ws6,[28,8,8,22,16,16,16,14,12])
print("   ✅ Sheet 6: Transport Analytics")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 7: PAYMENT STATUS
# ══════════════════════════════════════════════════════════════════════════════
ws7=wb.create_sheet("💳 Payment Status")
title_block(ws7,"Payment Status Tracker",
    "Payment status is trip-level. Revenue = Billed Amount summed per customer.",9)

c_paid  =df[df['Outward Code'].isin(paid_set)  ].groupby('Customer').agg(PT=('Outward Code','nunique'),PR=('Billed Amount (in Rs)','sum'))
c_unpaid=df[df['Outward Code'].isin(unpaid_set)].groupby('Customer').agg(UT=('Outward Code','nunique'),UR=('Billed Amount (in Rs)','sum'))
all_c=sorted(df['Customer'].unique())

hdrs7=["Customer","Paid Trips","Paid Revenue (₹)","Unpaid Trips","Unpaid Revenue (₹)","Total Trips","Total Revenue (₹)","% Paid"]
for ci,h in enumerate(hdrs7,1): ws7.cell(4,ci,h)
hdr_row(ws7,4,len(hdrs7))

for i,cname in enumerate(all_c,5):
    pt=int(c_paid.loc[cname,'PT'])   if cname in c_paid.index   else 0
    pr=c_paid.loc[cname,'PR']         if cname in c_paid.index   else 0
    ut=int(c_unpaid.loc[cname,'UT']) if cname in c_unpaid.index else 0
    ur=c_unpaid.loc[cname,'UR']       if cname in c_unpaid.index else 0
    tt=pt+ut; fill=C_UNPAID if ut>0 else rfill(i)
    wr(ws7,i,[cname,pt,pr,ut,ur,tt,pr+ur,pt/tt if tt else 0],
       fill,{3:INR,5:INR,7:INR,8:PCT})

tr7=5+len(all_c); total_row(ws7,tr7,len(hdrs7)); ws7.cell(tr7,1,"TOTAL")
for cl,fmt in [('B',INT),('C',INR),('D',INT),('E',INR),('F',INT),('G',INR)]:
    ci=ord(cl)-64
    ws7.cell(tr7,ci,f'=SUM({cl}5:{cl}{tr7-1})').number_format=fmt

# Unpaid detail list
dr=tr7+2; sec(ws7,dr,"⚠️ UNPAID TRANSACTION DETAIL",8,bg="FFD7D7",fg=C_RED_H); dr+=1
hdrs7u=["Date","Outward Code","Customer","Material","Type","Dispatched (KG)","Billed Rev (₹)","Transport Cost (₹)"]
for ci,h in enumerate(hdrs7u,1): ws7.cell(dr,ci,h)
hdr_row(ws7,dr,len(hdrs7u),bg=C_RED_H)

updf=(df[df['Outward Code'].isin(unpaid_set)]
        .sort_values('Date')
        [['Date','Outward Code','Customer','Material','Type',
          'Dispatched Qty (in KG)','Billed Amount (in Rs)','TotalTransportCost']])

for i,(_, rd) in enumerate(updf.iterrows()):
    rr=dr+1+i
    vals=[rd['Date'].strftime('%d %b %Y') if pd.notna(rd['Date']) else '',
          rd['Outward Code'],rd['Customer'],rd['Material'],rd['Type'],
          rd['Dispatched Qty (in KG)'],rd['Billed Amount (in Rs)'],rd['TotalTransportCost']]
    wr(ws7,rr,vals,"FFF5F5",{6:NUM,7:INR,8:INR})
    for ci in range(1,6): ws7.cell(rr,ci).alignment=_lft()

set_w(ws7,[36,14,36,28,12,16,18,18])
print("   ✅ Sheet 7: Payment Status")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 8: RATE & PRICING
# ══════════════════════════════════════════════════════════════════════════════
ws8=wb.create_sheet("📈 Rate & Pricing")
title_block(ws8,"Rate & Pricing Analysis",
    "Min / Avg / Max rate per material per customer — identify pricing inconsistencies.",9)

rate=(df[df['Rate (in Rs)']>0]
       .groupby(['Material','Customer'])
       .agg(Trips=('Outward Code','nunique'),
            MinR=('Rate (in Rs)','min'),
            AvgR=('Rate (in Rs)','mean'),
            MaxR=('Rate (in Rs)','max'),
            TotKG=('Dispatched Qty (in KG)','sum'),
            Billed=('Billed Amount (in Rs)','sum'))
       .reset_index().sort_values(['Material','AvgR'],ascending=[True,False]))
rate['Spread']=rate['MaxR']-rate['MinR']

hdrs8=["Material","Customer","# Trips","Min Rate (₹/KG)","Avg Rate (₹/KG)",
       "Max Rate (₹/KG)","Spread (₹)","Total KG","Billed Rev (₹)"]
for ci,h in enumerate(hdrs8,1): ws8.cell(4,ci,h)
hdr_row(ws8,4,len(hdrs8),bg=C_PURPLE)

prev_mat=""
for i,(_, rd) in enumerate(rate.iterrows()):
    rr=5+i
    # Highlight row when material changes
    fill=C_LBLUE if rd['Material']!=prev_mat else rfill(rr)
    prev_mat=rd['Material']
    wr(ws8,rr,[rd['Material'],rd['Customer'],rd['Trips'],rd['MinR'],rd['AvgR'],
               rd['MaxR'],rd['Spread'],rd['TotKG'],rd['Billed']],
       fill,{4:INR,5:INR,6:INR,7:INR,8:NUM,9:INR})
    # Flag where spread > 0 (inconsistent pricing)
    if rd['Spread']>0: ws8.cell(rr,7).fill=_fl("FFE0B2")

set_w(ws8,[32,36,8,16,16,16,14,14,18])
ws8.freeze_panes='A5'
print("   ✅ Sheet 8: Rate & Pricing")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 9: MONTHLY TRENDS
# ══════════════════════════════════════════════════════════════════════════════
ws9=wb.create_sheet("📅 Monthly Trends")
title_block(ws9,"Monthly Trend Analysis",
    "Volume & revenue: material-level sums. Transport & incentive: trip-deduped.",12)

mat_m=(df.groupby('MonthStr')
         .agg(Trips=('Outward Code','nunique'),
              DispKG=('Dispatched Qty (in KG)','sum'),
              AccKG=('AcceptedQty','sum'),
              RejKG=('Rejection Qty (in KG)','sum'),
              Billed=('Billed Amount (in Rs)','sum'),
              NetRev=('Value of Accepted Material (in Rs)','sum'),
              Custs=('Customer','nunique'))
         .reset_index())
trp_m=(df_trips.groupby(df_trips['Date'].dt.strftime('%b %Y'))
               .agg(TC=('TotalTransportCost','sum'),
                    IC=('Total Incentive Cost (in Rs)','sum'),
                    LVP=('LVP_Incentive','sum'))
               .reset_index().rename(columns={'Date':'MonthStr'}))
mon=mat_m.merge(trp_m,on='MonthStr',how='left').fillna(0)
mon['_d']=pd.to_datetime(mon['MonthStr'],format='%b %Y')
mon=mon.sort_values('_d').drop('_d',axis=1)
mon['RejPct']=mon['RejKG']/mon['DispKG'].replace(0,1)
mon['TCpctRev']=mon['TC']/mon['Billed'].replace(0,1)
mon['GrandRev']=mon['Billed']+mon['LVP']

hdrs9=["Month","# Trips","Dispatched (KG)","Accepted (KG)","Rejected (KG)",
       "Rej %","Billed Rev (₹)","LVP Incentive (₹)","Grand Rev (₹)",
       "Transport (₹)","Transp % of Billed","Unique Customers"]
for ci,h in enumerate(hdrs9,1): ws9.cell(4,ci,h)
hdr_row(ws9,4,len(hdrs9))

for i,(_, rd) in enumerate(mon.iterrows()):
    rr=5+i; fill=rfill(rr)
    wr(ws9,rr,[rd['MonthStr'],rd['Trips'],rd['DispKG'],rd['AccKG'],rd['RejKG'],
               rd['RejPct'],rd['Billed'],rd['LVP'],rd['GrandRev'],
               rd['TC'],rd['TCpctRev'],int(rd['Custs'])],
       fill,{3:NUM,4:NUM,5:NUM,6:PCT,7:INR,8:INR,9:INR,10:INR,11:PCT})

tr9=5+len(mon); total_row(ws9,tr9,len(hdrs9)); ws9.cell(tr9,1,"TOTAL / AVG")
for cl,fmt in [('B',INT),('C',NUM),('D',NUM),('E',NUM),
               ('G',INR),('H',INR),('I',INR),('J',INR)]:
    ci=ord(cl)-64
    ws9.cell(tr9,ci,f'=SUM({cl}5:{cl}{tr9-1})').number_format=fmt

set_w(ws9,[14,8,16,15,14,9,18,18,18,16,14,16])
print("   ✅ Sheet 9: Monthly Trends")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 10: TYPE ANALYSIS (Bags / Baled / Unprocessed)
# ══════════════════════════════════════════════════════════════════════════════
ws10=wb.create_sheet("📦 Type Analysis")
title_block(ws10,"Processing Type Analysis",
    "Compare Bags vs Baled vs Unprocessed — rate premium, volume and revenue by type.",8)

typ=(df[df['Rate (in Rs)']>0]
      .groupby(['Type','BroadCategory'])
      .agg(Trips=('Outward Code','nunique'),
           DispKG=('Dispatched Qty (in KG)','sum'),
           RejKG=('Rejection Qty (in KG)','sum'),
           RejPct=('Rejection %','mean'),
           AvgRate=('Rate (in Rs)','mean'),
           Billed=('Billed Amount (in Rs)','sum'))
      .reset_index().sort_values(['Type','Billed'],ascending=[True,False]))

hdrs10=["Type","Broad Category","# Trips","Dispatched (KG)","Rejected (KG)","Avg Rej %","Avg Rate (₹/KG)","Billed Rev (₹)"]
for ci,h in enumerate(hdrs10,1): ws10.cell(4,ci,h)
hdr_row(ws10,4,len(hdrs10))

prev_type=""; type_colors={"Bags":C_GREEN,"Baled":C_AMBER,"Unprocessed":C_ROSE}
for i,(_, rd) in enumerate(typ.iterrows()):
    rr=5+i
    fill=type_colors.get(rd['Type'], C_WHITE) if rd['Type']!=prev_type else rfill(rr)
    prev_type=rd['Type']
    wr(ws10,rr,[rd['Type'],rd['BroadCategory'],rd['Trips'],rd['DispKG'],rd['RejKG'],
                rd['RejPct']/100,rd['AvgRate'],rd['Billed']],
       fill,{4:NUM,5:NUM,6:PCT,7:INR,8:INR})

tr10=5+len(typ); total_row(ws10,tr10,len(hdrs10)); ws10.cell(tr10,1,"TOTAL")
for cl,fmt in [('C',INT),('D',NUM),('E',NUM),('H',INR)]:
    ci=ord(cl)-64
    ws10.cell(tr10,ci,f'=SUM({cl}5:{cl}{tr10-1})').number_format=fmt

# Type summary
sr10=tr10+2; sec(ws10,sr10,"TYPE SUMMARY",6); sr10+=1
for ci,h in enumerate(["Type","# Trips","Dispatched (KG)","Avg Rate (₹/KG)","Billed Rev (₹)","% of Revenue"],1):
    ws10.cell(sr10,ci,h)
hdr_row(ws10,sr10,6,bg=C_BLUE)
ts=(df[df['Rate (in Rs)']>0].groupby('Type')
      .agg(T=('Outward Code','nunique'),
           D=('Dispatched Qty (in KG)','sum'),
           AR=('Rate (in Rs)','mean'),
           B=('Billed Amount (in Rs)','sum'))
      .reset_index().sort_values('B',ascending=False))
tb10=ts['B'].sum()
for i,(_, rd) in enumerate(ts.iterrows()):
    rr=sr10+1+i; fill=type_colors.get(rd['Type'],C_WHITE)
    wr(ws10,rr,[rd['Type'],rd['T'],rd['D'],rd['AR'],rd['B'],rd['B']/tb10 if tb10 else 0],
       fill,{3:NUM,4:INR,5:INR,6:PCT})

set_w(ws10,[16,22,10,16,14,12,16,18])
print("   ✅ Sheet 10: Type Analysis")


# ══════════════════════════════════════════════════════════════════════════════
# SHEET 11: RAW DATA (cleaned)
# ══════════════════════════════════════════════════════════════════════════════
ws11=wb.create_sheet("📋 Raw Data")

ecols=['Date','Outward Code','Project','Customer','City','State',
       'Material','BroadCategory','Type','Vehicle number','VendorClean',
       'Driver Name','Dispatched Qty (in KG)','AcceptedQty',
       'Rejection Qty (in KG)','Rejection %','Rate (in Rs)',
       'Billed Amount (in Rs)','Value of Accepted Material (in Rs)',
       'TotalTransportCost','LVP_Incentive',
       'Incentive Cost/KG','MatIncentive','Payment Status']
ecols=[c for c in ecols if c in df.columns]

de=df[ecols].copy()
de['Date']=de['Date'].dt.strftime('%d %b %Y')
de['Payment Status']=de['Payment Status'].map({True:'Paid',False:'Unpaid'})

title_block(ws11,"Raw Transaction Data (Cleaned)",
    "Material-level rows. VendorClean=normalised name. BroadCategory=PDF category.",len(ecols))
for ci,h in enumerate(ecols,1): ws11.cell(4,ci,h)
hdr_row(ws11,4,len(ecols))

for i,(_, row) in enumerate(de.iterrows()):
    rr=5+i; fill=rfill(rr)
    for ci,v in enumerate(row.values,1):
        c=ws11.cell(rr,ci,v); c.fill=_fl(fill)
        c.border=_tb(); c.font=_df(); c.alignment=_lft()

for ci in range(1,len(ecols)+1):
    ws11.column_dimensions[get_column_letter(ci)].width=18
ws11.freeze_panes='A5'
print("   ✅ Sheet 11: Raw Data")


# ══════════════════════════════════════════════════════════════════════════════
# SAVE & DOWNLOAD
# ══════════════════════════════════════════════════════════════════════════════
wb.save(OUTPUT_FILENAME)
print(f"\n✅ Workbook saved: '{OUTPUT_FILENAME}'")
print("\nSheets:")
for i,s in enumerate(wb.sheetnames,1): print(f"  {i:>2}. {s}")

files.download(OUTPUT_FILENAME)
print("\n⬇️  Download started — check your browser downloads folder")
